# XModalix3D three-modality full-data alignment test

This notebook trains **XModalix3D** on the full paired NCCT, CBF and CBV datasets for 250 epochs and evaluates latent-space alignment.

The notebook is designed to be executed non-interactively on the HPC. Paths are taken from environment variables set by the accompanying SLURM script. No pipeline/model pickle is written.

**Important:** because `requires_paired=True`, the training set contains the intersection of sample IDs that are available in all three modalities.

## Environment and configuration

In [ ]:
import os
import sys
import inspect
from pathlib import Path

import torch
import autoencodix as acx

from autoencodix.configs import DataCase, DataConfig, DataInfo
from autoencodix.configs.xmodalix3d_config import XModalix3DConfig
from autoencodix.utils._losses import XModalLoss

DATA_ROOT = Path(
    os.environ.get(
        "DATA_ROOT",
        "/data/horse/ws/baeuchl-imagix3d/data/stroke_data",
    )
)

ncctpath = os.environ.get(
    "NCCT_PATH",
    str(DATA_ROOT / "ncct_flat"),
)
ncctannopath = os.environ.get(
    "NCCT_ANNO",
    str(DATA_ROOT / "ncct_anno_dst.csv"),
)
cbfpath = os.environ.get(
    "CBF_PATH",
    str(DATA_ROOT / "cbf_flat"),
)
cbfannopath = os.environ.get(
    "CBF_ANNO",
    str(DATA_ROOT / "cbf_anno_dst.csv"),
)
cbvpath = os.environ.get(
    "CBV_PATH",
    str(DATA_ROOT / "cbv_flat"),
)
cbvannopath = os.environ.get(
    "CBV_ANNO",
    str(DATA_ROOT / "cbv_anno_dst.csv"),
)

paths_to_check = {
    "NCCT volumes": Path(ncctpath),
    "NCCT annotation": Path(ncctannopath),
    "CBF volumes": Path(cbfpath),
    "CBF annotation": Path(cbfannopath),
    "CBV volumes": Path(cbvpath),
    "CBV annotation": Path(cbvannopath),
}

for name, path in paths_to_check.items():
    print(f"{name}: {path}")
    assert path.exists(), f"Missing input: {name}: {path}"

print()
print("Python:", sys.executable)
print("AUTOENCODIX:", acx.__file__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("CUDA device:", torch.cuda.get_device_name(0))

assert torch.cuda.is_available(), "This HPC notebook is configured for CUDA."

print("\nActive XModal adversarial-loss implementation:\n")
print(inspect.getsource(XModalLoss._calc_adversial_loss))

config = XModalix3DConfig(
    data_case=DataCase.IMG_TO_IMG,

    # 3D handling
    img_path_col="filepath",
    spatial_shape_policy="crop_or_pad_to_shape",
    target_shape_3d=(160, 192, 160),
    volume_scaling_strategy="per_volume",
    normalize_nonzero_only=True,

    # Full-data alignment run
    epochs=100,
    checkpoint_interval=5,
    batch_size=35,
    latent_dim=64,
    hidden_dim=16,
    pretrain_epochs=0,
    delta_class=0,

    reconstruction_loss="mse",
    loss_reduction="mean",
    scaling="NONE",
    learning_rate=0.00001,
    weight_decay=0.00001,
    beta=0.00001,
    anneal_function="logistic-late",

    # Reproducibility
    reproducible=True,
    global_seed=1,

    # XModalix alignment weights
    gamma=30,
    delta_pair=5,

    train_normalization="group",
    train_norm_groupsize=8,

    clamp_logvar=True,
    keep_mu_positive=False,

    requires_paired=True,
    device="cuda",
    n_gpus=1,

    data_config=DataConfig(
        data_info={
            "NCCT": DataInfo(
                file_path=ncctpath,
                extra_anno_file=ncctannopath,
                data_type="IMG",
                scaling="NONE",
                translate_direction="from",
                pretrain_epochs=0,
            ),
            "CBF": DataInfo(
                file_path=cbfpath,
                extra_anno_file=cbfannopath,
                data_type="IMG",
                scaling="NONE",
                translate_direction="to",
                pretrain_epochs=0,
            ),
            "CBV": DataInfo(
                file_path=cbvpath,
                extra_anno_file=cbvannopath,
                data_type="IMG",
                scaling="NONE",
                translate_direction=None,
                pretrain_epochs=0,
            ),
            "ANNO": DataInfo(
                file_path=ncctannopath,
                data_type="ANNOTATION",
            ),
        }
    ),
)

print("\nKey run configuration:")
for key in [
    "epochs",
    "checkpoint_interval",
    "batch_size",
    "latent_dim",
    "hidden_dim",
    "gamma",
    "delta_pair",
    "delta_class",
    "reproducible",
    "global_seed",
    "device",
]:
    print(f"{key}: {getattr(config, key)}")


## Memory-saving checkpoint capture

The generic XModal trainer normally stores full reconstructed 3D volumes at every checkpoint. With full-resolution volumes and a checkpoint every 5 epochs, retaining those reconstructions would consume very large amounts of RAM.

For this alignment experiment we only need the latent means (`mu`) and sample IDs at each checkpoint. The following local monkey patch changes **result capture only**; it does not alter the model forward pass, losses, gradients, optimizer, or training objective. The prediction step can still add the requested translation/reference reconstructions to the otherwise empty reconstruction dictionary.

In [ ]:
import numpy as np
from autoencodix.trainers._xmodal_trainer import XModalTrainer

def _capture_alignment_dynamics(
    self,
    batch_data,
):
    """
    Memory-saving checkpoint capture for the XModalix3D
    alignment experiment.

    Stores:
      - sampled latent z
      - latent mean mu
      - latent log-variance
      - sample IDs

    Does NOT store:
      - full 3D reconstructions
    """
    captured_data = {
        "latentspaces": {},
        "reconstructions": {},
        "mus": {},
        "sigmas": {},
        "sample_ids": {},
    }

    for mod_name, dynamics in self._modality_dynamics.items():

        sample_ids = batch_data[
            mod_name
        ].get("sample_ids")

        if sample_ids is not None:
            captured_data[
                "sample_ids"
            ][mod_name] = np.asarray(
                sample_ids
            )

        model_output = dynamics["mp"]

        # Sampled latent representation z
        if model_output.latentspace is not None:
            captured_data[
                "latentspaces"
            ][mod_name] = (
                model_output.latentspace.detach()
            )

        # Posterior mean mu
        if model_output.latent_mean is not None:
            captured_data[
                "mus"
            ][mod_name] = (
                model_output.latent_mean.detach()
            )

        # AUTOENCODIX stores latent_logvar
        # inside result.sigmas.
        if model_output.latent_logvar is not None:
            captured_data[
                "sigmas"
            ][mod_name] = (
                model_output.latent_logvar.detach()
            )

    return captured_data

XModalTrainer._capture_dynamics = (
    _capture_alignment_dynamics
)

print(
    "Patched XModalTrainer._capture_dynamics: "
    "stores z, mu, logvar and sample IDs; "
    "full-volume reconstructions are omitted."
)

## Preprocessing and integration checks

In [ ]:
from autoencodix.base._base_dataset import DataSetTypes
from autoencodix.data._xmodal3d_preprocessor import XModal3DPreprocessor
from autoencodix.modeling._volumevae_architecture import VolumeVAEArchitecture

xmodal3d = acx.XModalix3D(config=config)

xmodal3d.preprocess()

expected_modalities = {
    "img.NCCT",
    "img.CBF",
    "img.CBV",
}

for split_name in ["train", "valid", "test"]:
    split = getattr(
        xmodal3d._datasets,
        split_name,
    )

    print(f"\n--- {split_name} ---")
    print("modalities:", split.datasets.keys())
    print("fully paired:", split.is_fully_paired)

    assert set(split.datasets.keys()) == expected_modalities
    assert split.is_fully_paired

    id_sets = []

    for modality, ds in split.datasets.items():
        print(
            modality,
            "n =", len(ds),
            "shape =", tuple(ds[0][1].shape),
        )

        id_sets.append(set(ds.sample_ids))

    assert all(
        ids == id_sets[0]
        for ids in id_sets[1:]
    )

print("\nPreprocessing/pairing checks passed.")


## Inspect one multimodal batch

In [ ]:
from torch.utils.data import DataLoader
from autoencodix.data._multimodal_dataset import (
    create_multimodal_collate_fn,
)

train = xmodal3d._datasets.train

loader = DataLoader(
    train,
    batch_size=2,
    shuffle=False,
    collate_fn=create_multimodal_collate_fn(train),
)

batch = next(iter(loader))

assert set(batch.keys()) == expected_modalities

for modality, contents in batch.items():
    print(
        modality,
        contents["data"].shape,
        contents["sample_ids"],
    )


## Train XModalix3D

In [ ]:
xmodal3d.fit()

In [ ]:
from torch.utils.data import DataLoader
import numpy as np
import torch

from autoencodix.data._multimodal_dataset import (
    create_multimodal_collate_fn,
)


def extract_final_embeddings(
    trainer,
    dataset,
    batch_size,
):
    """
    Extract mu, z, logvar and sample IDs from one split
    using the final trained models in eval mode.

    No shuffling, no dropped samples.
    """
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        drop_last=False,
        collate_fn=create_multimodal_collate_fn(
            multimodal_dataset=dataset
        ),
    )

    mu_parts = {
        mod: []
        for mod in dataset.datasets.keys()
    }

    z_parts = {
        mod: []
        for mod in dataset.datasets.keys()
    }

    logvar_parts = {
        mod: []
        for mod in dataset.datasets.keys()
    }

    id_parts = {
        mod: []
        for mod in dataset.datasets.keys()
    }

    for dynamics in trainer._modality_dynamics.values():
        dynamics["model"].eval()

    with torch.inference_mode():

        for batch in loader:

            for mod_name, mod_data in batch.items():

                model = (
                    trainer
                    ._modality_dynamics[
                        mod_name
                    ]["model"]
                )

                x = mod_data["data"].to(
                    trainer._fabric.device
                )

                output = model(x)

                mu_parts[
                    mod_name
                ].append(
                    output.latent_mean
                    .detach()
                    .cpu()
                    .numpy()
                )

                z_parts[
                    mod_name
                ].append(
                    output.latentspace
                    .detach()
                    .cpu()
                    .numpy()
                )

                logvar_parts[
                    mod_name
                ].append(
                    output.latent_logvar
                    .detach()
                    .cpu()
                    .numpy()
                )

                id_parts[
                    mod_name
                ].extend(
                    mod_data["sample_ids"]
                )

    mu_dict = {
        mod: np.concatenate(parts, axis=0)
        for mod, parts in mu_parts.items()
    }

    z_dict = {
        mod: np.concatenate(parts, axis=0)
        for mod, parts in z_parts.items()
    }

    logvar_dict = {
        mod: np.concatenate(parts, axis=0)
        for mod, parts in logvar_parts.items()
    }

    id_dict = {
        mod: np.asarray(ids).astype(str)
        for mod, ids in id_parts.items()
    }

    return (
        mu_dict,
        z_dict,
        logvar_dict,
        id_dict,
    )

In [ ]:
final_train_mu, final_train_z, final_train_logvar, final_train_ids = (
    extract_final_embeddings(
        trainer=xmodal3d._trainer,
        dataset=xmodal3d._datasets.train,
        batch_size=config.batch_size,
    )
)

final_valid_mu, final_valid_z, final_valid_logvar, final_valid_ids = (
    extract_final_embeddings(
        trainer=xmodal3d._trainer,
        dataset=xmodal3d._datasets.valid,
        batch_size=config.batch_size,
    )
)

final_test_mu, final_test_z, final_test_logvar, final_test_ids = (
    extract_final_embeddings(
        trainer=xmodal3d._trainer,
        dataset=xmodal3d._datasets.test,
        batch_size=config.batch_size,
    )
)

## Predict NCCT → CBF on the held-out test split

In [ ]:
xmodal3d.predict(
    from_key="NCCT",
    to_key="CBF",
)

## Alignment helper functions

In [ ]:
import numpy as np
import pandas as pd

from itertools import combinations

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.metrics import (
    balanced_accuracy_score,
    confusion_matrix,
)


def get_aligned_pair(
    latent_dict,
    id_dict,
    modality_a,
    modality_b,
):
    ids_a = np.asarray(id_dict[modality_a]).astype(str)
    ids_b = np.asarray(id_dict[modality_b]).astype(str)

    z_a = np.asarray(latent_dict[modality_a])
    z_b = np.asarray(latent_dict[modality_b])

    idx_a = {sid: i for i, sid in enumerate(ids_a)}
    idx_b = {sid: i for i, sid in enumerate(ids_b)}

    common_ids = sorted(set(ids_a) & set(ids_b))

    A = np.stack([z_a[idx_a[sid]] for sid in common_ids])
    B = np.stack([z_b[idx_b[sid]] for sid in common_ids])

    return common_ids, A, B


def pairwise_alignment_report_exact(
    latent_dict,
    id_dict,
):
    """Compare matched cross-modal distances with all unmatched patient pairs."""
    modalities = sorted(latent_dict.keys())
    rows = []

    for a, b in combinations(modalities, 2):
        common_ids, A, B = get_aligned_pair(
            latent_dict,
            id_dict,
            a,
            b,
        )

        matched_distances = np.linalg.norm(
            A - B,
            axis=1,
        )
        matched_mean = matched_distances.mean()

        distance_matrix = np.linalg.norm(
            A[:, None, :] - B[None, :, :],
            axis=2,
        )

        off_diagonal_mask = ~np.eye(
            len(common_ids),
            dtype=bool,
        )

        unmatched_distances = distance_matrix[
            off_diagonal_mask
        ]
        unmatched_mean = unmatched_distances.mean()

        rows.append({
            "modality_a": a,
            "modality_b": b,
            "n_pairs": len(common_ids),
            "matched_distance": matched_mean,
            "unmatched_distance": unmatched_mean,
            "matched/unmatched_ratio": (
                matched_mean / unmatched_mean
                if unmatched_mean > 0
                else np.nan
            ),
        })

    return pd.DataFrame(rows)


def build_modality_probe_data(
    latent_dict,
    id_dict,
):
    X = []
    y = []
    groups = []

    for modality, z in latent_dict.items():
        z = np.asarray(z)
        ids = np.asarray(id_dict[modality]).astype(str)

        for row, sid in zip(z, ids):
            X.append(row)
            y.append(modality)
            groups.append(sid)

    return (
        np.asarray(X),
        np.asarray(y),
        np.asarray(groups),
    )


def modality_probe_score(
    latent_dict,
    id_dict,
):
    X, y, groups = build_modality_probe_data(
        latent_dict,
        id_dict,
    )

    modalities = sorted(np.unique(y))
    n_groups = len(np.unique(groups))

    cv = GroupKFold(
        n_splits=min(5, n_groups)
    )

    probe = make_pipeline(
        StandardScaler(),
        LogisticRegression(
            max_iter=5000,
            class_weight="balanced",
        ),
    )

    pred = cross_val_predict(
        probe,
        X,
        y,
        cv=cv,
        params={"groups": groups},
    )

    score = balanced_accuracy_score(
        y,
        pred,
    )
    chance = 1.0 / len(modalities)

    cm = confusion_matrix(
        y,
        pred,
        labels=modalities,
    )

    cm_df = pd.DataFrame(
        cm,
        index=modalities,
        columns=modalities,
    )

    return score, chance, cm_df


## Final training and test alignment

In [ ]:
available_epochs = sorted(
    epoch
    for epoch, split_data
    in xmodal3d.result.mus._data.items()
    if (
        split_data is not None
        and "train" in split_data
        and split_data["train"] is not None
    )
)

assert available_epochs, "No stored training checkpoints were found."

final_epoch = max(available_epochs)

print("Final stored internal epoch:",final_epoch)
print("Human-readable epoch:",final_epoch + 1)


# --------------------------------------------------
# TRAIN representations
# --------------------------------------------------

train_mu = xmodal3d.result.mus.get(
    epoch=final_epoch,
    split="train",
)

train_z = xmodal3d.result.latentspaces.get(
    epoch=final_epoch,
    split="train",
)

train_logvar = xmodal3d.result.sigmas.get(
    epoch=final_epoch,
    split="train",
)

train_ids = xmodal3d.result.sample_ids.get(
    epoch=final_epoch,
    split="train",
)


# --------------------------------------------------
# TEST representations
# --------------------------------------------------

test_mu = xmodal3d.result.mus.get(
    epoch=-1,
    split="test",
)

test_z = xmodal3d.result.latentspaces.get(
    epoch=-1,
    split="test",
)

test_logvar = xmodal3d.result.sigmas.get(
    epoch=-1,
    split="test",
)

test_ids = xmodal3d.result.sample_ids.get(
    epoch=-1,
    split="test",
)


# --------------------------------------------------
# Pairwise alignment: mu
# --------------------------------------------------

print("\nTRAIN pairwise alignment — MU:")

train_mu_alignment = (
    pairwise_alignment_report_exact(
        train_mu,
        train_ids,
    )
)

display(train_mu_alignment)


print("\nTEST pairwise alignment — MU:")

test_mu_alignment = (
    pairwise_alignment_report_exact(
        test_mu,
        test_ids,
    )
)

display(test_mu_alignment)


# --------------------------------------------------
# Pairwise alignment: z
# --------------------------------------------------

print("\nTRAIN pairwise alignment — Z:")

train_z_alignment = (
    pairwise_alignment_report_exact(
        train_z,
        train_ids,
    )
)

display(train_z_alignment)


print("\nTEST pairwise alignment — Z:")

test_z_alignment = (
    pairwise_alignment_report_exact(
        test_z,
        test_ids,
    )
)

display(test_z_alignment)

## Final modality-probe diagnostics

In [ ]:
# --------------------------------------------------
# MU probes
# --------------------------------------------------

train_mu_probe, chance, train_mu_cm = (
    modality_probe_score(
        train_mu,
        train_ids,
    )
)

test_mu_probe, test_chance, test_mu_cm = (
    modality_probe_score(
        test_mu,
        test_ids,
    )
)


print(
    "TRAIN MU balanced modality accuracy:",
    train_mu_probe,
)
print("Chance:", chance)
display(train_mu_cm)


print(
    "\nTEST MU balanced modality accuracy:",
    test_mu_probe,
)
print("Chance:", test_chance)
display(test_mu_cm)


# --------------------------------------------------
# Z probes
# --------------------------------------------------

train_z_probe, chance_z, train_z_cm = (
    modality_probe_score(
        train_z,
        train_ids,
    )
)

test_z_probe, test_chance_z, test_z_cm = (
    modality_probe_score(
        test_z,
        test_ids,
    )
)


print(
    "\nTRAIN Z balanced modality accuracy:",
    train_z_probe,
)
print("Chance:", chance_z)
display(train_z_cm)


print(
    "\nTEST Z balanced modality accuracy:",
    test_z_probe,
)
print("Chance:", test_chance_z)
display(test_z_cm)

In [ ]:
for split_name, mu_dict, z_dict, id_dict in [
    (
        "train",
        final_train_mu,
        final_train_z,
        final_train_ids,
    ),
    (
        "valid",
        final_valid_mu,
        final_valid_z,
        final_valid_ids,
    ),
    (
        "test",
        final_test_mu,
        final_test_z,
        final_test_ids,
    ),
]:

    mu_score, chance, _ = modality_probe_score(
        mu_dict,
        id_dict,
    )

    z_score, _, _ = modality_probe_score(
        z_dict,
        id_dict,
    )

    print(
        split_name,
        "mu probe:",
        mu_score,
        "z probe:",
        z_score,
        "chance:",
        chance,
    )

In [ ]:
for split_name, mu_dict, z_dict, id_dict in [
    (
        "train",
        final_train_mu,
        final_train_z,
        final_train_ids,
    ),
    (
        "valid",
        final_valid_mu,
        final_valid_z,
        final_valid_ids,
    ),
    (
        "test",
        final_test_mu,
        final_test_z,
        final_test_ids,
    ),
]:

    print(
        f"\n{split_name.upper()} MU"
    )

    display(
        pairwise_alignment_report_exact(
            mu_dict,
            id_dict,
        )
    )

    print(
        f"\n{split_name.upper()} Z"
    )

    display(
        pairwise_alignment_report_exact(
            z_dict,
            id_dict,
        )
    )

## Latent distribution diagnostics

In [ ]:
def print_latent_statistics(
    split_name,
    mu_dict,
    z_dict,
    logvar_dict,
):
    print(f"\n===== {split_name.upper()} =====")

    for modality in sorted(mu_dict.keys()):

        mu = np.asarray(
            mu_dict[modality]
        )

        z = np.asarray(
            z_dict[modality]
        )

        logvar = np.asarray(
            logvar_dict[modality]
        )

        # Posterior standard deviation:
        # sigma = exp(0.5 * logvar)
        sigma = np.exp(
            0.5 * logvar
        )

        variance = np.exp(
            logvar
        )

        print(f"\n{modality}")

        print(
            "mean latent MU std across dimensions:",
            np.std(
                mu,
                axis=0,
            ).mean(),
        )

        print(
            "mean sampled Z std across dimensions:",
            np.std(
                z,
                axis=0,
            ).mean(),
        )

        print(
            "mean logvar:",
            logvar.mean(),
        )

        print(
            "mean posterior sigma:",
            sigma.mean(),
        )

        print(
            "mean posterior variance:",
            variance.mean(),
        )


print_latent_statistics(
    "train",
    train_mu,
    train_z,
    train_logvar,
)

print_latent_statistics(
    "test",
    test_mu,
    test_z,
    test_logvar,
)

## Alignment across training checkpoints

In [ ]:
print("Stored checkpoint epochs:",available_epochs,)
print("Human-readable training epochs:",[epoch + 1 for epoch in available_epochs])

probe_rows = []
pair_rows = []

for epoch in available_epochs:

    checkpoint_ids = (
        xmodal3d.result.sample_ids.get(
            epoch=epoch,
            split="train",
        )
    )

    checkpoint_mu = (
        xmodal3d.result.mus.get(
            epoch=epoch,
            split="train",
        )
    )

    checkpoint_z = (
        xmodal3d.result.latentspaces.get(
            epoch=epoch,
            split="train",
        )
    )

    # ==========================================
    # MU
    mu_probe, chance, _ = (
        modality_probe_score(
            checkpoint_mu,
            checkpoint_ids,
        )
    )

    probe_rows.append({
        "epoch_internal": epoch,
        "epoch": epoch + 1,
        "representation": "mu",
        "probe_balanced_accuracy":
            mu_probe,
        "chance": chance,
    })


    mu_pair_report = (
        pairwise_alignment_report_exact(
            checkpoint_mu,
            checkpoint_ids,
        )
    )

    mu_pair_report[ "epoch_internal"] = epoch

    mu_pair_report["epoch"] = epoch + 1

    mu_pair_report["representation"] = "mu"

    pair_rows.append(
        mu_pair_report
    )

    # ==========================================
    # Z

    z_probe, chance_z, _ = (
        modality_probe_score(
            checkpoint_z,
            checkpoint_ids,
        )
    )

    probe_rows.append({
        "epoch_internal": epoch,
        "epoch": epoch + 1,
        "representation": "z",
        "probe_balanced_accuracy":
            z_probe,
        "chance": chance_z,
    })


    z_pair_report = (
        pairwise_alignment_report_exact(
            checkpoint_z,
            checkpoint_ids,
        )
    )

    z_pair_report["epoch_internal"] = epoch

    z_pair_report["epoch"] = epoch + 1

    z_pair_report["representation"] = "z"

    pair_rows.append(z_pair_report)


probe_over_time = pd.DataFrame(probe_rows)

pairwise_over_time = pd.concat(
    pair_rows,
    ignore_index=True,
)


display(probe_over_time)

display(
    pairwise_over_time[
        [
            "epoch",
            "representation",
            "modality_a",
            "modality_b",
            "matched_distance",
            "unmatched_distance",
            "matched/unmatched_ratio",
        ]
    ]
)

## Plot modality-separability

In [ ]:
import matplotlib.pyplot as plt


plt.figure(figsize=(9, 5))


for representation, group in (
    probe_over_time.groupby(
        "representation"
    )
):

    plt.plot(
        group["epoch"],
        group[
            "probe_balanced_accuracy"
        ],
        marker="o",
        label=representation,
    )


plt.axhline(
    y=probe_over_time[
        "chance"
    ].iloc[0],
    linestyle="--",
    label="Chance",
)


plt.xlabel("Training epoch")

plt.ylabel(
    "Balanced modality accuracy"
)

plt.title(
    "Modality separability: mu versus z"
)

plt.legend()
plt.show()

## Patient-alignment plot

In [ ]:
for representation in ["mu", "z"]:

    plt.figure(figsize=(9, 5))

    subset = pairwise_over_time[
        pairwise_over_time[
            "representation"
        ] == representation
    ]

    for (
        modality_a,
        modality_b,
    ), group in subset.groupby(
        [
            "modality_a",
            "modality_b",
        ]
    ):

        plt.plot(
            group["epoch"],
            group[
                "matched/unmatched_ratio"
            ],
            marker="o",
            label=(
                f"{modality_a} ↔ "
                f"{modality_b}"
            ),
        )


    plt.axhline(
        y=1.0,
        linestyle="--",
        label=(
            "No patient-specific alignment"
        ),
    )

    plt.xlabel("Training epoch")

    plt.ylabel(
        "Matched / unmatched distance"
    )

    plt.title(
        "Patient-specific alignment "
        f"using {representation}"
    )

    plt.legend()

    plt.show()

## XModalix loss components across training

In [ ]:
def get_sub_loss_value(
    key,
    epoch,
    split="train",
):
    """
    Retrieve one scalar XModalix sub-loss value.
    """
    dynamics = (
        xmodal3d.result.sub_losses.get(
            key
        )
    )

    if dynamics is None:
        return np.nan

    value = dynamics.get(
        epoch=epoch,
        split=split,
    )

    if value is None:
        return np.nan

    value = np.asarray(value)

    return float(
        value.squeeze()
    )


modalities = [
    "img.NCCT",
    "img.CBF",
    "img.CBV",
]


loss_rows = []


for epoch in range(
    config.epochs
):

    row = {
        "epoch": epoch + 1,
    }


    # ------------------------------------------
    # XModal losses
    # ------------------------------------------

    row["adver_loss"] = (
        get_sub_loss_value(
            "adver_loss",
            epoch,
        )
    )

    row["paired_loss"] = (
        get_sub_loss_value(
            "paired_loss",
            epoch,
        )
    )

    row["clf_loss"] = (
        get_sub_loss_value(
            "clf_loss",
            epoch,
        )
    )


    # ------------------------------------------
    # Per-modality VAE losses
    # ------------------------------------------

    recon_values = []
    var_values = []
    beta_values = []
    anneal_values = []


    for modality in modalities:

        recon = get_sub_loss_value(
            f"{modality}.recon_loss",
            epoch,
        )

        var = get_sub_loss_value(
            f"{modality}.var_loss",
            epoch,
        )

        beta = get_sub_loss_value(
            f"{modality}.effective_beta_factor",
            epoch,
        )

        anneal = get_sub_loss_value(
            f"{modality}.anneal_factor",
            epoch,
        )


        row[
            f"{modality}.recon_loss"
        ] = recon

        row[
            f"{modality}.var_loss"
        ] = var

        row[
            f"{modality}.effective_beta_factor"
        ] = beta


        recon_values.append(
            recon
        )

        var_values.append(
            var
        )

        beta_values.append(
            beta
        )

        anneal_values.append(
            anneal
        )


    row[
        "mean_recon_loss"
    ] = np.nanmean(
        recon_values
    )

    row[
        "mean_var_loss"
    ] = np.nanmean(
        var_values
    )

    row[
        "mean_effective_beta_factor"
    ] = np.nanmean(
        beta_values
    )

    row[
        "mean_anneal_factor"
    ] = np.nanmean(
        anneal_values
    )


    loss_rows.append(row)


loss_over_time = pd.DataFrame(
    loss_rows
)

display(loss_over_time)

## Loss plots

**Alignment losses**

In [ ]:
plt.figure(figsize=(9, 5))

for column in [
    "adver_loss",
    "paired_loss",
    "clf_loss",
]:
    plt.plot(
        loss_over_time["epoch"],
        loss_over_time[column],
        label=column,
    )

plt.xlabel("Training epoch")
plt.ylabel("Loss")
plt.title(
    "XModalix alignment-loss components"
)
plt.legend()
plt.show()

**Reconstruction and variational contribution**

In [ ]:
plt.figure(figsize=(9, 5))

plt.plot(
    loss_over_time["epoch"],
    loss_over_time[
        "mean_recon_loss"
    ],
    label="Mean reconstruction loss",
)

plt.plot(
    loss_over_time["epoch"],
    loss_over_time[
        "mean_var_loss"
    ],
    label="Mean variational loss",
)

plt.xlabel("Training epoch")
plt.ylabel("Loss")
plt.title(
    "VAE reconstruction and "
    "variational loss"
)

plt.legend()
plt.show()

**beta/annealing plot**

In [ ]:
plt.figure(figsize=(9, 5))

plt.plot(
    loss_over_time["epoch"],
    loss_over_time[
        "mean_effective_beta_factor"
    ],
    label="Effective beta factor",
)

plt.plot(
    loss_over_time["epoch"],
    loss_over_time[
        "mean_anneal_factor"
    ],
    label="Anneal factor",
)

plt.xlabel("Training epoch")
plt.ylabel("Factor")

plt.title(
    "KL annealing across training"
)

plt.legend()
plt.show()

**posterior logvar/sigma trajectories**

In [ ]:
posterior_rows = []


for epoch in available_epochs:

    logvar_dict = (
        xmodal3d.result.sigmas.get(
            epoch=epoch,
            split="train",
        )
    )

    if not isinstance(
        logvar_dict,
        dict,
    ):
        continue


    for modality, logvar in (
        logvar_dict.items()
    ):

        logvar = np.asarray(
            logvar
        )

        sigma = np.exp(
            0.5 * logvar
        )

        variance = np.exp(
            logvar
        )


        posterior_rows.append({
            "epoch":
                epoch + 1,

            "modality":
                modality,

            "mean_logvar":
                logvar.mean(),

            "mean_sigma":
                sigma.mean(),

            "mean_variance":
                variance.mean(),
        })


posterior_over_time = (
    pd.DataFrame(
        posterior_rows
    )
)

display(
    posterior_over_time
)

**plot posterior sigma**

In [ ]:
plt.figure(figsize=(9, 5))


for modality, group in (
    posterior_over_time.groupby(
        "modality"
    )
):

    plt.plot(
        group["epoch"],
        group["mean_sigma"],
        marker="o",
        label=modality,
    )


plt.xlabel("Training epoch")

plt.ylabel(
    "Mean posterior sigma"
)

plt.title(
    "Posterior uncertainty across training"
)

plt.legend()

plt.show()

**plot log-variance**

In [ ]:
plt.figure(figsize=(9, 5))


for modality, group in (
    posterior_over_time.groupby(
        "modality"
    )
):

    plt.plot(
        group["epoch"],
        group["mean_logvar"],
        marker="o",
        label=modality,
    )


plt.xlabel("Training epoch")

plt.ylabel(
    "Mean latent log-variance"
)

plt.title(
    "Posterior log-variance across training"
)

plt.legend()

plt.show()

**\(\mu\)-modality accuracy, \(z\)-modality accuracy, effective beta**

In [ ]:
fig, ax1 = plt.subplots(
    figsize=(10, 5)
)


for representation, group in (
    probe_over_time.groupby(
        "representation"
    )
):

    ax1.plot(
        group["epoch"],
        group[
            "probe_balanced_accuracy"
        ],
        marker="o",
        label=(
            f"Probe accuracy ({representation})"
        ),
    )


ax1.axhline(
    1 / 3,
    linestyle="--",
    label="Chance",
)

ax1.set_xlabel(
    "Training epoch"
)

ax1.set_ylabel(
    "Balanced modality accuracy"
)


ax2 = ax1.twinx()

ax2.plot(
    loss_over_time["epoch"],
    loss_over_time[
        "mean_effective_beta_factor"
    ],
    linestyle=":",
    label="Effective beta",
)

ax2.set_ylabel(
    "Effective beta factor"
)


lines1, labels1 = (
    ax1.get_legend_handles_labels()
)

lines2, labels2 = (
    ax2.get_legend_handles_labels()
)

ax1.legend(
    lines1 + lines2,
    labels1 + labels2,
    loc="best",
)

plt.title(
    "Modality separability versus KL annealing"
)

plt.show()